### Imports

In [4]:
# Imports
import re
import nltk
import numpy as np
import pandas as pd
from google.colab import drive

nltk.download('punkt', force=True)
nltk.download('punkt_tab', force=True)
nltk.download('popular', force=True)

from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

import collections
from collections import defaultdict

from sklearn.model_selection import train_test_split
import pickle
import os

import copy

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from google.colab import drive
import os

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package gazetteers to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/genesis.zip.
[nltk_data]    | Downloading package gutenberg to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gutenberg.zip.
[nltk_data]    | Downloading package inaugural to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/inaugural.zip.
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzip

### Upload data

In [5]:
# Get data
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/Datasets/NLP_v_0_datasets/test.csv"
df = pd.read_csv(file_path)

Mounted at /content/drive


### Text Preprocessing

##### Help Preprocessing Functions

In [6]:
def transform_word_to_index(word, dictionary):
  """Transforms a word into an index using a dictionary"""
  try:
    return dictionary[word]
  except KeyError:
    return dictionary['<UNK>']

In [7]:
def process_abbreviations(tokens):
    return [t.replace('.', '') if '.' in t else t for t in tokens]

In [8]:
def get_length_list(sequences):
  length_list = []
  length = []

  for seqs in sequences:
    for seq in seqs:
      length.append(len(seq))

    # length.sort()
    length_list.append(length)
    length = []

  return length_list

In [9]:
def max_sentences_len(text):
  return max(len(sent) for sent in text)

In [10]:
def print_nested(data, indent=0):
    if isinstance(data, list):
        if any(isinstance(i, list) for i in data):
            print(' ' * indent + '[')
            for item in data:
                print_nested(item, indent + 4)
            print(' ' * indent + ']')
        else:
            print(' ' * indent + str(data))
    else:
        print(' ' * indent + str(data))

In [11]:
def create_unified_word_index(texts, summaries, min_word_freq=1, max_vocab_size=None):
    """Creates a single dictionary for texts and summaries"""
    all_texts = texts + summaries
    word2idx, idx2word = create_word_index(all_texts, min_word_freq, max_vocab_size)

    return word2idx, idx2word

##### Main Preprocessing Functions

In [12]:
def normalize_abbreviations(text):
    """Finds all letter sequences with dots between them"""
    return re.sub(r'\b((?:[A-Za-z]\.)+[A-Za-z])\b',
                 lambda m: m.group(0).replace('.', ''),
                 text)

def create_word_index(texts, min_word_freq=1, max_vocab_size=None):
  """
  Create word2idx & idx2word with special tokens for multiple texts
  """
  # Initialization
  word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
  idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}

  word_counts = defaultdict(int)


  # Text handling
  for text in texts:
    # Clearing
    clear_text = normalize_abbreviations(text)
    clear_text = re.sub(r'[^\w\s.!?]', ' ', clear_text)
    clear_text = ' '.join(clear_text.split()).lower()

    # Tokenizing
    tokens = word_tokenize(clear_text)

    # Counting
    for word in tokens:
      word_counts[word] += 1


  # Sorting by count
  sorted_words = sorted(word_counts.items(),
                      key=lambda x: (-x[1], x[0]))


  # Dictionary formation
  for word, count in sorted_words:
    if count >= min_word_freq:
      if max_vocab_size and len(word2idx) >= max_vocab_size:
        break
      if word not in word2idx:
        idx = len(word2idx)
        word2idx[word] = idx
        idx2word[idx] = word

  return word2idx, idx2word

In [13]:
def text_to_sequence(text, dictionary, max_len):
  """
  Converts raw text into a sequence of numeric indexes
  replacing unknown words with <UNK>
  """
  seqs = []

  clear_text = re.sub(r'[^\w\s?.!]', '', text)
  clear_text = ' '.join(clear_text.split()).lower()

  tokens = word_tokenize(clear_text)
  tokens = process_abbreviations(tokens)

  if len(tokens) > max_len -2:
    tokens = tokens[:max_len - 2]

  return [dictionary['<SOS>'],
          *[transform_word_to_index(t, dictionary) for t in tokens],
          dictionary['<EOS>']]

  # sents = sent_tokenize(clear_text)

  # for sent in sents:
  #   # Pipeline for processing text sentences
  #   tokens = word_tokenize(sent)

  #   # 2 - number of start/end special tokens
  #   if len(tokens) > max_len:
  #     del tokens[max_len-2:]

  #   tokens = process_abbreviations(tokens)

  #   seq = [transform_word_to_index(t, dictionary) for t in tokens]

  #   final_seq = [dictionary['<SOS>']] + seq + [dictionary['<EOS>']]

  #   # Split into batches
  #   seqs.append(final_seq)
  #   # seqs += final_seq

  # return seqs

In [14]:
def texts_to_sequence(texts, dictionary, max_len):
  """Converts a list of texts using `text_to_sequence()`"""
  # Initialization
  # text_seqs = []

  # for text in texts:
  #   text_seqs.append(text_to_sequence(text, dictionary, max_len))

  # return text_seqs
  return [text_to_sequence(text,dictionary, max_len) for text in texts]

In [15]:
def create_batches(sequences, pad_idx, batch_size=4):
  batches = []

  for i in range(0, len(sequences), batch_size):
    batch_sequences = sequences[i:i+batch_size]
    tensor_sequences = [torch.tensor(seq) for seq in batch_sequences]
    padded_batch = torch.nn.utils.rnn.pad_sequence(
        tensor_sequences,
        batch_first=True,
        padding_value=pad_idx
    )
    batches.append(padded_batch)

  return batches

  # for text in texts_sequences:
  #   for j in range(0, len(text), batch_size):
  #     # Separate
  #     batch = text[j:j+batch_size]

  #     # Add 0's
  #     # [0] - can be replaced with the <PAD> value from the dictionary (word_to_index['<PAD>'])
  #     padded_batch = [sent + [dictionary['<PAD>']]*(max_sentences_len(batch) - len(sent)) for sent in batch]

  #     # Batches for one text
  #     batches.append(padded_batch)

  #   # List of batched texts
  #   texts_bathes.append(batches)
  #   batches = []

  # return texts_bathes

In [16]:
def replace_non_zero_numpy(data):
    arr = np.array(data)
    return np.where(arr != 0, 1, 0).tolist()

def generate_mask(texts_bathes):
  return [[replace_non_zero_numpy(inner) for inner in middle] for middle in texts_bathes]

# def generate_mask(texts_batches):
#   return [
#       [
#           [[1 if token != 0 else 0 for token in sent] for sent in batch]
#           for batch in text_batches
#       ]
#       for text_batches in texts_batches
#   ]

In [17]:
def save_dictionaries(word2idx, idx2word, path):
  try:
      with open(path, 'x') as f:
        pass
  except FileExistsError:
      print("File already exist!")

  dictionaries = {
      'word2idx': word2idx,
      'idx2word': idx2word
  }

  with open(path, 'wb') as file:
    pickle.dump(dictionaries, file)

In [18]:
def open_loaded_pickle(path):
  with open(path, 'rb') as file:
    return pickle.load(file)

In [19]:
def train_val_test_split(X, y):
  X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42
  )

  X_train, X_val, y_train, y_val = train_test_split(
      X_temp, y_temp,
      test_size=0.1765,
      random_state=42
  )

  return X_train, X_val, X_test, y_train, y_val, y_test

### Dataset Functions

In [20]:
class TextDataset(Dataset):
  def __init__(self, source_texts, target_texts, pad_idx=0):
    """
    Parameters:
      source_texts: list[list[int]] - articles in the form of index sequences
      target_texts: list[list[int]] - summaries in the form of index sequences
      pad_idx: - padding token index
    """
    self.source_texts = source_texts
    self.target_texts = target_texts
    self.pad_idx = pad_idx

    assert len(self.source_texts) == len(self.target_texts)

  def __len__(self):
    """Returns total number of samples"""
    return len(self.source_texts)

  def __getitem__(self, idx):
    """Returns one non-padding tensor by index"""
    return {
        'src': torch.tensor(self.source_texts[idx], dtype=torch.long),
        'tgt': torch.tensor(self.source_texts[idx], dtype=torch.long)
    }

In [21]:
def collate_fn(batch, pad_idx):
  """
  Combines several samples into a batch
  Called automatically from DataLoader
  """
  src_sequences = [item['src'] for item in batch] # list[tensor]
  tgt_sequences = [item['tgt'] for item in batch] # list[tensor]

  src_padded = torch.nn.utils.rnn.pad_sequence(
      src_sequences,
      batch_first=True,
      padding_value=pad_idx
  )

  tgt_padded = torch.nn.utils.rnn.pad_sequence(
      tgt_sequences,
      batch_first=True,
      padding_value=pad_idx
  )

  return src_padded, tgt_padded

### Model

##### Model Help Functions

In [22]:
def create_src_mask(src, pad_idx=0):
  """Create padding mask: [batch_size, 1, 1, seq_len]"""
  src_mask = (src == pad_idx).unsqueeze(1).unsqueeze(2)
  return src_mask


def create_tgt_mask(tgt, pad_idx=0):
  """
  Combines several samples into a batch
  Called automatically from DataLoader
  """
  # !!! Possible optimized aproach !!!
  batch_size, seq_len = tgt.shape

  # Padding Mask
  pad_mask = (tgt == pad_idx).unsqueeze(1).unsqueeze(2)

  # Future Mask
  future_mask = torch.tril(
      torch.ones(seq_len, seq_len, device=tgt.device, dtype=torch.bool)
  ).unsqueeze(0).unsqueeze(0)

  combined_mask = pad_mask | ~future_mask

  return combined_mask

In [23]:
def create_padding_mask(seq, pad_idx):
    """
    Input: sequence [batch_size, seq_len], pad_idx
    Output: mask [batch_size, 1, 1, seq_len] где 1 для pad токенов
    """
    return (seq == pad_idx).unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(size):
    """
    Input: size (длина последовательности)
    Output: верхняя треугольная матрица для предотвращения look-ahead
    """
    return torch.triu(torch.ones(size, size), diagonal=1).bool()

##### Model Functions

In [24]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, dropout):
    super().__init__()

    # dropout
    self.dropout = nn.Dropout(p=dropout)

    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                         (-torch.log(torch.tensor(10000.0)) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    # register buffer
    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return self.dropout(x)

In [25]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, dropout):
    super().__init__()
    assert d_model % num_heads == 0

    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.d_v = d_model // num_heads

    self.linear_q = nn.Linear(d_model, d_model)
    self.linear_k = nn.Linear(d_model, d_model)
    self.linear_v = nn.Linear(d_model, d_model)
    self.linear_out = nn.Linear(d_model, d_model)

    # softmax stabilization
    self.scale = torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32)) # on CPU fix in the future

    self.attention_weights = None

    self.dropout = nn.Dropout(dropout)

  def forward(self, q, k, v, mask=None):
    batch_size, seq_len, _ = q.shape

    q = self.linear_q(q)
    k = self.linear_k(k)
    v = self.linear_v(v)

    q = q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
    k = k.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
    v = v.view(batch_size, -1, self.num_heads, self.d_v).transpose(1,2)

    scores = torch.matmul(q, k.transpose(-2,-1))

    scores /= self.scale

    # mask
    if mask is not None:
      scores = scores.masked_fill(mask == 0, -1e9)

    attn_weights = F.softmax(scores, dim=-1) # -1 - softmax on keys
    attn_weights = self.dropout(attn_weights)

    context = torch.matmul(attn_weights, v) # dimension: (b, h, seq_len_q, d_v)

    context = context.transpose(1, 2)
    context = context.contiguous().view(batch_size, -1, self.d_model)

    output = self.linear_out(context)
    output = self.dropout(output)

    return output, attn_weights

In [26]:
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff = None, dropout = 0.1, activation = 'relu'):
    super().__init__()

    if d_ff is None:
      d_ff = d_model * 4

    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

    if activation == 'relu':
      self.activation = nn.ReLU()
    elif activation == 'gelu':
      self.activation = nn.GELU()
    else:
      raise ValueError(f"Unsupported activation function: {activation}")

  def forward(self, x):
    x = self.linear1(x) # (batch_size, seq_len, d_model) -> (batch_size, seq_len, d_ff)
    x = self.activation(x) # apply neural function
    x = self.dropout(x)
    x = self.linear2(x) # (b, s, d_ff) -> (b, s, d_model)
    return x

In [27]:
class TransformerBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff = None, dropout = 0.1):
    super().__init__()

    if d_ff is None:
      d_ff = d_model * 4

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    residual = x
    attn_output, attn_weights = self.self_attn(x, x, x, mask) # q = k = v = x
    self.attention_weights = attn_weights
    attn_output = self.dropout1(attn_output)
    x = residual + attn_output
    x = self.norm1(x)

    residual = x
    ff_output = self.feed_forward(x)
    ff_output = self.dropout2(ff_output)
    x = residual + ff_output
    x = self.norm2(x)

    return x

In [28]:
class Decoder(nn.Module):
  """Composition of several decoding layers and application of final normalization"""
  def __init__(self, decoder_layer, num_layers, d_model):
    super().__init__()
    # creates a stack of N independent but architecturally identical
    # decoder layers, each with its own unique weights
    self.layers = nn.ModuleList([
        copy.deepcopy(decoder_layer) for _ in range(num_layers)
    ])

    self.norm = nn.LayerNorm(d_model)

  def forward(self, tgt, enc_output, tgt_mask=None, src_mask=None):
    x = tgt
    for layer in self.layers:
      x = layer(x, enc_output, tgt_mask, src_mask)

    return self.norm(x)

In [29]:
class DecoderLayer(nn.Module):
  """Logic of one processing step (self-attn -> cross-attn -> FFN)"""
  def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
    super().__init__()

    if d_ff is None:
      d_ff = d_model * 4

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)

    self.feed_forward = FeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)
    self.dropout3 = nn.Dropout(dropout)

  def forward(self, x, enc_output, tgt_mask, src_mask):
    # Self-attention
    residual = x
    attn1, _ = self.self_attn(x, x, x, mask=tgt_mask)
    x = residual + self.dropout1(attn1)
    x = self.norm1(x)

    # Cross-attention
    residual = x
    # q(x) from decoder; k,v(enc_output) from encoder
    attn2, cross_weights = self.cross_attn(x, enc_output, enc_output, mask=src_mask)
    self.cross_attention_weights = cross_weights
    x = residual + self.dropout2(attn2)
    x = self.norm2(x)

    # Feed forward
    residual = x
    ff = self.feed_forward(x)
    x = residual + self.dropout3(ff)
    x = self.norm3(x)

    return x

In [30]:
class Encoder(nn.Module):
  """Composition of multiple EncoderLayers and application of final normalization"""
  def __init__(self, encoder_layer, num_layers, d_model):
    super().__init__()
    self.layers = nn.ModuleList([
        copy.deepcopy(encoder_layer) for _ in range(num_layers)
    ])

    self.norm = nn.LayerNorm(d_model)

  def forward(self, x, src_mask):
    output = x
    for layer in self.layers:
      output = layer(output, src_mask)

    return self.norm(output)

In [31]:
class EncoderLayer(nn.Module):
  """Logic of one processing step (self-attn -> FFN)"""
  def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
    super().__init__()

    if d_ff is None:
      d_ff = d_model * 4

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, x, src_mask):
    residual = x
    attn_output, _ = self.self_attn(x, x, x, mask=src_mask)
    x = residual + self.dropout1(attn_output)
    x = self.norm1(x)

    residual = x
    ff_output = self.feed_forward(x)
    x = residual + self.dropout2(ff_output)
    x = self.norm2(x)

    return x

In [32]:
class Seq2SeqTransformer(nn.Module):
  def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads,
               num_encoder_layers, num_decoder_layers, dropout, pad_idx, max_len=5000, d_ff=None):
    super().__init__()

    self.pad_idx = pad_idx
    self.d_model = d_model

    self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
    self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)

    # Инициализация эмбеддингов
    nn.init.normal_(self.src_embed.weight, mean=0, std=d_model**-0.5)
    nn.init.normal_(self.tgt_embed.weight, mean=0, std=d_model**-0.5)
    nn.init.constant_(self.src_embed.weight[pad_idx], 0)
    nn.init.constant_(self.tgt_embed.weight[pad_idx], 0)

    self.positional_encoding = PositionalEncoding(
        d_model=d_model,
        max_len=max_len,
        dropout=dropout
    )

    encoder_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)
    self.encoder = Encoder(encoder_layer, num_encoder_layers, d_model)

    decoder_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)
    self.decoder = Decoder(decoder_layer, num_decoder_layers, d_model)

    self.generator = nn.Linear(d_model, tgt_vocab_size)

  def forward(self, src, tgt):
    src_mask = create_src_mask(src, self.pad_idx)
    tgt_mask = create_tgt_mask(tgt, self.pad_idx)


  # src_emb = self.src_embed(src) * math.sqrt(self.d_model) ???
  # tgt_emb = self.tgt_embed(tgt) * math.sqrt(self.d_model) ???
    src_emb = self.positional_encoding(self.src_embed(src))
    tgt_emb = self.positional_encoding(self.tgt_embed(tgt))

    enc_output = self.encoder(src_emb, src_mask)

    dec_output = self.decoder(tgt_emb, enc_output, tgt_mask, src_mask)

    output = self.generator(dec_output)

    return output

### Train Functions

In [69]:
def setup_google_drive():
  """Connect to Google Drive"""
  drive.mount('/content/drive')
  # Создаем папку для моделей в Google Drive
  model_dir = '/content/drive/MyDrive/colab_notebooks/model_checkpoints'
  os.makedirs(model_dir, exist_ok=True)
  return model_dir

In [37]:
def train_epoch(model, dataloader, optimizer, criterion, device):
  total_loss = 0
  for batch in tqdm(dataloader, desc='Training'):
    src, tgt = batch
    src, tgt = src.to(device), tgt.to(device)

    # Forward pass
    optimizer.zero_grad()
    output = model(src, tgt[:, :-1])

    # Calculate loss
    loss = criterion(
      output.view(-1, output.size(-1)),
      tgt[:, 1:].contiguous().view(-1)
    )

    # Backward pass
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [38]:
def evaluate(model, dataloader, criterion, device):
  total_loss = 0
  with torch.no_grad():
    for batch in tqdm(dataloader, desc='Validation'):
      src, tgt = batch
      src, tgt = src.to(device), tgt.to(device)

      output = model(src, tgt[:, :-1])
      loss = criterion(
        output.view(-1, output.size(-1)),
        tgt[:, 1:].contiguous().view(-1)
      )

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [39]:
def train_model(model, train_loader, val_loader, epochs, device, pad_idx, criterion):
  # Setup Google Drive
  drive_dir = setup_google_drive()

  # Optim and Loss
  optimizer = torch.optim.Adam(
      model.parameters(),
      lr=0.0001,
      betas=(0.9, 0.98),
      eps=1e-9
  )
  # criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

  local_checkpoints_dir = 'checkpoints'
  os.makedirs(local_checkpoint_dir, exist_ok=True)

  # Checkpoints
  # os.makedirs('checkpoints', exist_ok=True)
  best_val_loss = float('inf')

  # Learning rate scheduler (optional)
  scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95) # *

  # Lists to store loss history
  train_losses = []
  val_losses = []

  drive_model_path = os.path.join(drive_dir, 'best_model.pt')
  if os.path.exists(drive_model_path):
      print("Found existing model in Google Drive. Loading...")
      checkpoint = torch.load(drive_model_path, map_location=device)
      model.load_state_dict(checkpoint['model_state_dict'])
      optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
      scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
      best_val_loss = checkpoint['val_loss']
      train_losses = checkpoint.get('train_losses', [])
      val_losses = checkpoint.get('val_losses', [])
      start_epoch = checkpoint['epoch'] + 1
      print(f"Resumed from epoch {checkpoint['epoch']}, best val_loss: {best_val_loss:.4f}")
  else:
      start_epoch = 0
      print("No existing model found in Google Drive. Starting fresh training.")

  # Train Loop
  for epoch in range(epochs):
    # Train epoch
    model.train()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)

    # Validation epoch
    model.eval()
    val_loss = evaluate(model, val_loader, criterion, device)

    # Step scheduler
    scheduler.step() # *

    # Store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
      best_val_loss = val_loss

      checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': val_loss,
        'train_loss': train_loss,
        'train_losses': train_losses,
        'val_losses': val_losses
      }

      # Сохраняем в Google Drive
      torch.save(checkpoint, drive_model_path)

      # Также сохраняем локально на всякий случай
      local_model_path = os.path.join(local_checkpoint_dir, f'best_model_epoch_{epoch}.pt')
      torch.save(checkpoint, local_model_path)

      print(f'New best model saved to Google Drive with val_loss: {val_loss:.4f}')

      # Сохраняем последнюю модель каждую эпоху
      last_checkpoint = {
          'epoch': epoch,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'val_loss': val_loss,
          'train_loss': train_loss,
          'train_losses': train_losses,
          'val_losses': val_losses
      }

      last_model_path = os.path.join(drive_dir, 'last_model.pt')
      torch.save(last_checkpoint, last_model_path)

      print(f'Epoch {epoch}: Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}')

    print(f"Training completed! Best model saved in: {drive_model_path}")
    return train_losses, val_losses

In [40]:
def train_model_with_drive(model, train_loader, val_loader, epochs, device, pad_idx, criterion):
    # Настройка Google Drive
    drive_dir = setup_google_drive()

    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.0001,
        betas=(0.9, 0.98),
        eps=1e-9
    )

    # Локальная папка для чекпоинтов (на время сессии)
    local_checkpoint_dir = 'checkpoints'
    os.makedirs(local_checkpoint_dir, exist_ok=True)

    best_val_loss = float('inf')
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)

    train_losses = []
    val_losses = []

    # Проверяем, есть ли уже сохраненная модель в Google Drive
    drive_model_path = os.path.join(drive_dir, 'best_model.pt')
    if os.path.exists(drive_model_path):
        print("Found existing model in Google Drive. Loading...")
        checkpoint = torch.load(drive_model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        best_val_loss = checkpoint['val_loss']
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed from epoch {checkpoint['epoch']}, best val_loss: {best_val_loss:.4f}")
    else:
        start_epoch = 0
        print("No existing model found in Google Drive. Starting fresh training.")

    # Train Loop
    for epoch in range(start_epoch, start_epoch + epochs):
        # Train epoch
        model.train()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)

        # Validation epoch
        model.eval()
        val_loss = evaluate(model, val_loader, criterion, device)

        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Сохраняем лучшую модель в Google Drive
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_loss': val_loss,
                'train_loss': train_loss,
                'train_losses': train_losses,
                'val_losses': val_losses
            }

            # Сохраняем в Google Drive
            torch.save(checkpoint, drive_model_path)

            # Также сохраняем локально на всякий случай
            local_model_path = os.path.join(local_checkpoint_dir, f'best_model_epoch_{epoch}.pt')
            torch.save(checkpoint, local_model_path)

            print(f'New best model saved to Google Drive with val_loss: {val_loss:.4f}')

        # Сохраняем последнюю модель каждую эпоху
        last_checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': val_loss,
            'train_loss': train_loss,
            'train_losses': train_losses,
            'val_losses': val_losses
        }

        last_model_path = os.path.join(drive_dir, 'last_model.pt')
        torch.save(last_checkpoint, last_model_path)

        print(f'Epoch {epoch}: Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}')

    print(f"Training completed! Best model saved in: {drive_model_path}")
    return train_losses, val_losses

In [70]:
# Функция для загрузки модели из Google Drive
def load_model_from_drive(model, device, model_type='best'):
    """
    Загружает модель из Google Drive

    Args:
        model: Инициализированная модель
        device: Устройство (cuda/cpu)
        model_type: 'best' или 'last'
    """
    drive_dir = '/content/drive/MyDrive/colab_notebooks/model_checkpoints'
    model_path = os.path.join(drive_dir, f'{model_type}_model.pt')

    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Model loaded from {model_path}")
        print(f"Epoch: {checkpoint['epoch']}, Val Loss: {checkpoint['val_loss']:.4f}")
        return model, checkpoint
    else:
        print(f"No {model_type} model found at {model_path}")
        return model, None

In [42]:
# Функция для продолжения обучения
def continue_training_from_drive(model, train_loader, val_loader, additional_epochs, device, pad_idx, criterion):
    """
    Продолжает обучение с последней сохраненной модели из Google Drive
    """
    # Загружаем последнюю модель
    model, checkpoint = load_model_from_drive(model, device, 'last')

    if checkpoint is None:
        print("No model found to continue from. Starting fresh training.")
        return train_model_with_drive(model, train_loader, val_loader, additional_epochs, device, pad_idx, criterion)

    # Используем функцию train_model_with_drive для продолжения
    return train_model_with_drive(model, train_loader, val_loader, additional_epochs, device, pad_idx, criterion)

### Summary

##### Summary methods

In [43]:
def greedy_decode(model, src, max_len=50, temperature=1.0):
  """Choose token with max probability"""
  model.eval()
  with torch.no_grad():
    # Create mask
    src_mask = create_src_mask(src, model.pad_idx)

    # Distinct
    src_emb = model.positional_encoding(model.src_embed(src))
    memory = model.encoder(src_emb, src_mask)

    # Initialize output with <SOS> token ?
    ys = torch.ones(1, 1).fill_(1).type_as(src.data)

    for i in range(max_len - 1):
      # Create mask for decoder
      tgt_mask = create_tgt_mask(ys, model.pad_idx)

      #  through decoder - ?
      tgt_emb = model.positional_encoding(model.tgt_embed(ys))
      out = model.decoder(tgt_emb, memory, tgt_mask, src_mask)
      logits = model.generator(out[:, -1, :])

      # Temperature sampling
      logits = logits / temperature
      prob = F.softmax(logits, dim=-1)

      # Choose next token - ?
      _, next_word = torch.max(prob, dim=1)
      next_word = next_word.item()
      ys = torch.cat([ys,
                      torch.ones(1, 1).type_as(src.data).fill_(next_word)],
                      dim = 1)

      if next_word == 2: # <EOS>
        break

    return ys

In [44]:
def beam_search_decode(model, src, beam_size=5, max_len=50):
  """Look at some variants"""
  model.eval()
  with torch.no_grad():
    src_mask = create_src_mask(src, model.pad_idx)
    src_emb = model.positional_encoding(model.src_embed(src))
    memory = model.encoder(src_emb, src_mask)

    # Initialize beam
    beams = [([1], 0)] # (sequence, score)

    for step in range(max_len):
      all_candidates = []

      for seq, score in beams:
        if seq[-1] == 2: # <EOS>
          all_candidates.append((seq, score))
          continue

        # Preparing input for decoder
        ys = torch.tensor(seq).unsqueeze(0).type_as(src.data)
        tgt_mask = create_tgt_mask(ys, model.pad_idx)

        tgt_emb = model.positional_encoding(model.tgt_embed(ys))
        out = model.decoder(tgt_emb, memory, tgt_mask, src_mask)
        logits = model.generator(out[:, -1, :])
        probabilities = F.log_softmax(logits, dim=-1)

        # Take top-k candidates
        topk_probs, topk_indices = torch.topk(probabilities, beam_size, dim=1)

        for i in range(beam_size):
          next_token = topk_indices[0, i].item()
          next_score = score + topk_probs[0, i].item()
          new_seq = seq + [next_token]
          all_candidates.append((new_seq, next_score))

      # Sort and choose the best beam_size candidates
      all_candidates.sort(key=lambda x: x[1], reverse=True)
      beams = all_candidates[:beam_size]

      # Check, if all sequences end
      if all(seq[-1] == 2 for seq, score in beams):
        break

    # Return the best sequence
    best_seq, best_score = beams[0]
    return torch.tensor(best_seq).unsqueeze(0)

##### Summarizer

In [45]:
class TextSummarizer():
  def __init__(self, model, word2idx, idx2word, device, max_length=128):
    self.model = model
    self.word2idx = word2idx
    self.idx2word = idx2word
    self.device = device
    self.max_length = max_length
    self.model.eval()

  def preprocess_text(self, text):
    """Preprocessing text into index sequence"""

    # Clearing text
    text = re.sub(r'[^\w\s\.\?\!]', '', text)
    text = ' '.join(text.split()).lower()

    # Tokenizing
    tokens = word_tokenize(text)
    tokens = [t.replace('.', '') if '.' in t else t for t in tokens]

    # Cutting by max length
    if len(tokens) > self.max_length - 2:
      tokens = tokens[:self.max_length - 2]

    # Transformation into indexes
    sequence = [self.word2idx['<SOS>']]
    sequence.extend([self.word2idx.get(token, self.word2idx['<UNK>'])
                    for token in tokens])
    sequence.append(self.word2idx['<EOS>'])

    return sequence

  def tokens_to_text(self, tokens):
    """Converting tokens back to text"""
    # Cleaning up special tokens
    filtered_tokens = [token for token in tokens
                       if token not in [self.word2idx['<SOS>'],
                                        self.word2idx['<EOS>'],
                                        self.word2idx['<PAD>']]]

    # Converting indexes into words
    words = [self.idx2word.get(idx, '<UNK>') for idx in filtered_tokens]
    return ' '.join(words)

  def summarize(self, text, method='greedy', beam_size=3, temperature=0.8, max_len=50):
    """
    Main Function

    Args:
      text: source texxt for summarization
      method: 'greedy' or 'beam'
      beam_size: beam size for beam search
      temperature: parameter for temperature sampling
      max_len: max length of summary
    """
    # Preprocessig
    src_sequence = self.preprocess_text(text)
    src_tensor = torch.tensor(src_sequence).unsqueeze(0).to(self.device)

    # Generating in relation to method
    if method == 'greedy':
      output_tokens = greedy_decode(self.model, src_tensor, max_len, temperature)
    elif method == 'beam':
      output_tokens = beam_search_decode(self.model, src_tensor, beam_size, max_len)
    else:
      raise ValueError("Method must be 'greedy' or 'beam'")

    # Converting back to text
    summary_tokens = output_tokens.squeeze(0).cpu().tolist() # cpu - ?
    summary_text = self.tokens_to_text(summary_tokens)

    return summary_text

  def batch_summarize(self, texts, batch_size=8, **kwargs):
    """Summarizing the batch of texts"""
    summarirs = []
    for i in range(0, len(texts), batch_size):
      batch_texts = texts[i:i + batch_size]
      batch_summaries = [self.summarize(text, **kwargs) for text in batch_texts]
      summaries.extend(batch_summaries)
    return summaries

### Checking

In [46]:
def analyze_generation_issues(model, dataloader, word2idx, idx2word, device, num_samples=3):
    """Анализ проблем генерации"""
    model.eval()
    summarizer = TextSummarizer(model, word2idx, idx2word, device)

    print("=== ANALYZING GENERATION ISSUES ===")

    for i, (src_batch, tgt_batch) in enumerate(dataloader):
        if i >= num_samples:
            break

        # Переносим на устройство
        src = src_batch[0:1].to(device)  # Берем первый пример батча и переносим на device
        tgt = tgt_batch[0:1].to(device)

        # Декодируем исходные данные
        src_text = summarizer.tokens_to_text(src[0].cpu().tolist())
        tgt_text = summarizer.tokens_to_text(tgt[0].cpu().tolist())

        print(f"\n--- Sample {i+1} ---")
        print(f"Source: {src_text[:100]}...")
        print(f"Target: {tgt_text}")

        # Анализируем распределение вероятностей
        with torch.no_grad():
            src_mask = create_src_mask(src, model.pad_idx)
            src_emb = model.positional_encoding(model.src_embed(src))
            memory = model.encoder(src_emb, src_mask)

            # Генерируем пошагово с анализом
            generated = [1]  # <SOS>

            for step in range(10):  # Только первые 10 шагов
                tgt_seq = torch.tensor(generated, device=device).unsqueeze(0)  # Сразу создаем на device
                tgt_mask = create_tgt_mask(tgt_seq, model.pad_idx)

                tgt_emb = model.positional_encoding(model.tgt_embed(tgt_seq))
                output = model.decoder(tgt_emb, memory, tgt_mask, src_mask)
                logits = model.generator(output[:, -1, :])
                probs = F.softmax(logits, dim=-1)

                # Топ-5 предсказаний
                topk_probs, topk_indices = torch.topk(probs, 5)

                print(f"Step {step}:")
                for j, (prob, idx) in enumerate(zip(topk_probs[0], topk_indices[0])):
                    word = idx2word.get(idx.item(), '<UNK>')
                    print(f"    {prob.item():.4f} - '{word}'")

                next_token = torch.argmax(probs, dim=-1).item()
                generated.append(next_token)

                if next_token == 2:
                    break

        generated_text = summarizer.tokens_to_text(generated)
        print(f"Generated: {generated_text}")

In [47]:
def improved_beam_search_decode(model, src, beam_size=5, max_len=50,
                              length_penalty=0.8, min_length=5):
    """
    Улучшенный beam search с length penalty и другими оптимизациями
    """
    model.eval()
    with torch.no_grad():
        src_mask = create_src_mask(src, model.pad_idx)
        src_emb = model.positional_encoding(model.src_embed(src))
        memory = model.encoder(src_emb, src_mask)

        # Инициализация beam
        beams = [([1], 0, False)]  # (sequence, score, finished)

        for step in range(max_len):
            all_candidates = []

            for seq, score, finished in beams:
                if finished:
                    all_candidates.append((seq, score, finished))
                    continue

                if seq[-1] == 2:  # <EOS>
                    all_candidates.append((seq, score, True))
                    continue

                # Подготовка входа для декодера
                ys = torch.tensor(seq).unsqueeze(0).type_as(src.data)
                tgt_mask = create_tgt_mask(ys, model.pad_idx)

                tgt_emb = model.positional_encoding(model.tgt_embed(ys))
                out = model.decoder(tgt_emb, memory, tgt_mask, src_mask)
                logits = model.generator(out[:, -1, :])
                log_probs = F.log_softmax(logits, dim=-1)

                # Берем top-k кандидатов
                topk_log_probs, topk_indices = torch.topk(log_probs, beam_size * 2, dim=1)

                for i in range(beam_size * 2):
                    next_token = topk_indices[0, i].item()
                    token_log_prob = topk_log_probs[0, i].item()

                    # Length penalty
                    current_length = len(seq) + 1
                    length_penalty_factor = ((5 + current_length) / (5 + 1)) ** length_penalty

                    new_score = score + token_log_prob / length_penalty_factor
                    new_seq = seq + [next_token]

                    # Проверяем минимальную длину для EOS
                    is_finished = (next_token == 2 and current_length >= min_length)

                    all_candidates.append((new_seq, new_score, is_finished))

            # Сортируем и выбираем лучшие
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beams = all_candidates[:beam_size]

            # Проверяем, все ли последовательности завершены
            if all(finished for _, _, finished in beams):
                break

        # Возвращаем лучшую последовательность
        best_seq, best_score, _ = beams[0]
        return torch.tensor(best_seq).unsqueeze(0)

In [48]:
def improved_greedy_decode(model, src, max_len=50, temperature=0.8, top_k=20):
    """
    Улучшенный greedy decode с top-k фильтрацией
    """
    model.eval()
    with torch.no_grad():
        src_mask = create_src_mask(src, model.pad_idx)
        src_emb = model.positional_encoding(model.src_embed(src))
        memory = model.encoder(src_emb, src_mask)

        ys = torch.ones(1, 1).fill_(1).type_as(src.data)  # <SOS>

        for i in range(max_len - 1):
            tgt_mask = create_tgt_mask(ys, model.pad_idx)

            tgt_emb = model.positional_encoding(model.tgt_embed(ys))
            out = model.decoder(tgt_emb, memory, tgt_mask, src_mask)
            logits = model.generator(out[:, -1, :])

            # Top-k фильтрация
            if top_k > 0:
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits[indices_to_remove] = -float('Inf')

            # Temperature sampling
            logits = logits / max(temperature, 0.1)
            probs = F.softmax(logits, dim=-1)

            # Выбираем следующий токен
            next_token = torch.multinomial(probs, 1).item()

            ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_token)], dim=1)

            if next_token == 2 and i > 5:  # EOS только после минимальной длины
                break

        return ys

In [49]:
class ImprovedTextSummarizer(TextSummarizer):
    def __init__(self, model, word2idx, idx2word, device, max_length=512):
        super().__init__(model, word2idx, idx2word, device, max_length)

    def improved_summarize(self, text, method='beam', beam_size=3, temperature=0.8,
                         max_len=100, length_penalty=0.8, top_k=20):
        """
        Улучшенная функция суммаризации
        """
        # Препроцессинг с сохранением больше контекста
        src_sequence = self.preprocess_text(text)
        src_tensor = torch.tensor(src_sequence).unsqueeze(0).to(self.device)

        # Выбор метода генерации
        if method == 'greedy':
            output_tokens = improved_greedy_decode(
                self.model, src_tensor, max_len, temperature, top_k
            )
        elif method == 'beam':
            output_tokens = improved_beam_search_decode(
                self.model, src_tensor, beam_size, max_len, length_penalty
            )
        else:
            raise ValueError("Method must be 'greedy' or 'beam'")

        # Пост-обработка
        summary_tokens = output_tokens.squeeze(0).cpu().tolist()
        summary_text = self.post_process_summary(summary_tokens)

        return summary_text

    def post_process_summary(self, tokens):
        """Пост-обработка сгенерированного текста"""
        # Убираем служебные токены
        filtered_tokens = [token for token in tokens
                          if token not in [self.word2idx['<SOS>'],
                                         self.word2idx['<EOS>'],
                                         self.word2idx['<PAD>']]]

        # Преобразуем в слова
        words = [self.idx2word.get(idx, '') for idx in filtered_tokens]

        # Убираем повторяющиеся слова подряд
        cleaned_words = []
        prev_word = None
        for word in words:
            if word != prev_word:
                cleaned_words.append(word)
            prev_word = word

        # Ограничиваем максимальную длину
        if len(cleaned_words) > 50:
            cleaned_words = cleaned_words[:50]

        return ' '.join(cleaned_words)

    def compare_summaries(self, texts, methods=['greedy', 'beam']):
        """Сравнение разных методов генерации"""
        for i, text in enumerate(texts[:3]):  # Первые 3 текста
            print(f"\n=== Text {i+1} ===")
            print(f"Input: {text[:100]}...")

            for method in methods:
                summary = self.improved_summarize(text, method=method)
                print(f"{method.upper()}: {summary}")

In [50]:
def evaluate_model_performance(model, dataloader, word2idx, idx2word, device):
    """Оценка производительности модели"""
    summarizer = ImprovedTextSummarizer(model, word2idx, idx2word, device)

    references = []
    hypotheses = []

    print("Evaluating model performance...")

    with torch.no_grad():
        for i, (src_batch, tgt_batch) in enumerate(dataloader):
            if i >= 10:  # Ограничим для скорости
                break

            for src, tgt in zip(src_batch, tgt_batch):
                # Исходный текст
                src_text = summarizer.tokens_to_text(src.cpu().tolist())

                # Эталонная суммаризация
                ref_text = summarizer.tokens_to_text(tgt.cpu().tolist())
                references.append(ref_text)

                # Сгенерированная суммаризация
                hyp_text = summarizer.improved_summarize(src_text, method='beam')
                hypotheses.append(hyp_text)

                print(f"\nSample {i}:")
                print(f"Source: {src_text[:80]}...")
                print(f"Reference: {ref_text}")
                print(f"Generated: {hyp_text}")

    return references, hypotheses

In [51]:
def test_diversity(summarizer, sample_texts):
    """Тестирование на разных текстах для проверки адаптации"""
    print("=== TESTING DIVERSITY ===")

    for i, text in enumerate(sample_texts):
        summary1 = summarizer.improved_summarize(text, method='greedy')
        summary2 = summarizer.improved_summarize(text, method='beam')

        print(f"\n--- Text {i+1} ---")
        print(f"Input: {text[:80]}...")
        print(f"Greedy: {summary1}")
        print(f"Beam: {summary2}")

        # Проверяем, что суммаризации разные для разных текстов
        if i > 0:
            prev_summary = summarizer.improved_summarize(sample_texts[i-1], method='beam')
            if summary2 == prev_summary:
                print("⚠️  WARNING: Same summary for different texts!")

### Main Pipeline

##### Text Preprocessing

In [52]:
# Upload dataset
texts = df['article']
summaries = df['highlights']
sum = texts + summaries

In [53]:
text_for_summary = df['article'][0]

In [54]:
print(texts)

0        Ever noticed how plane seats appear to be gett...
1        A drunk teenage boy had to be rescued by secur...
2        Dougie Freedman is on the verge of agreeing a ...
3        Liverpool target Neto is also wanted by PSG an...
4        Bruce Jenner will break his silence in a two-h...
                               ...                        
11485    Our young Earth may have collided with a body ...
11486    A man facing trial for helping his former love...
11487    A dozen or more metal implements are arranged ...
11488    Brook Lopez dominated twin brother Robin with ...
11489    A Chinese hospital is being painstakingly move...
Name: article, Length: 11490, dtype: object


In [55]:
# Create dictionaries
word_2_idx, idx_2_word = create_unified_word_index(texts, summaries, 2, 70000)
len(word_2_idx)

67901

In [56]:
# Create sequences
X = texts_to_sequence(texts, word_2_idx, 64) # texts in numbers
y = texts_to_sequence(summaries, word_2_idx, 64)

In [57]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)

##### Dataset & DataLoader

In [58]:
# Choose device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [59]:
# Create Dataset
train_dataset = TextDataset(
    source_texts = X_train,
    target_texts = y_train,
    pad_idx=word_2_idx['<PAD>']
)

val_dataset = TextDataset(
    source_texts = X_val,
    target_texts = y_val,
    pad_idx=word_2_idx['<PAD>']
)

In [60]:
# Create DataLoader
train_loader = DataLoader(
  train_dataset,
  batch_size=32,
  shuffle=32,
  collate_fn=lambda batch: collate_fn(batch, word_2_idx['<PAD>']),
  num_workers=2,
  pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
  val_dataset,
  batch_size=32,
  shuffle=False,
  collate_fn=lambda batch: collate_fn(batch, word_2_idx['<PAD>']),
  num_workers=2
)

##### Model Initializing

In [61]:
# Parameters
src_vocab_size = len(word_2_idx)
tgt_vocab_size = len(word_2_idx)
pad_idx = word_2_idx['<PAD>']

# Create Model
model = Seq2SeqTransformer(
  src_vocab_size=src_vocab_size,
  tgt_vocab_size=tgt_vocab_size,
  d_model=512,
  num_heads=8,
  num_encoder_layers=3,
  num_decoder_layers=3,
  dropout=0.1,
  pad_idx=pad_idx,
  max_len=5000,
  d_ff=2048
)

# To device
model = model.to(device)

##### Launching training

In [62]:
# criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# epochs = 10

# train_losses, val_losses = train_model(
#   model=model,
#   train_loader=train_loader,
#   val_loader=val_loader,
#   epochs=epochs,
#   device=device,
#   pad_idx=pad_idx,
#   criterion=criterion
# )

In [72]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Первоначальное обучение (автоматически найдет существующую модель если есть)
train_losses, val_losses = train_model_with_drive(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=2,
    device=device,
    pad_idx=pad_idx,
    criterion=criterion
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No existing model found in Google Drive. Starting fresh training.


Validation: 100%|██████████| 54/54 [00:04<00:00, 12.01it/s]


New best model saved to Google Drive with val_loss: 5.9358
Epoch 0: Train Loss 6.8170, Val Loss 5.9358


Validation: 100%|██████████| 54/54 [00:04<00:00, 12.13it/s]


New best model saved to Google Drive with val_loss: 4.8489
Epoch 1: Train Loss 5.4636, Val Loss 4.8489
Training completed! Best model saved in: /content/drive/MyDrive/colab_notebooks/model_checkpoints/best_model.pt


In [73]:
# Загрузка модели для инференса
model, checkpoint = load_model_from_drive(model, device, 'best')

Model loaded from /content/drive/MyDrive/colab_notebooks/model_checkpoints/best_model.pt
Epoch: 1, Val Loss: 4.8489


In [82]:
# Продолжение обучения
train_losses, val_losses = continue_training_from_drive(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    additional_epochs=3,
    device=device,
    pad_idx=pad_idx,
    criterion=criterion
)

Model loaded from /content/drive/MyDrive/colab_notebooks/model_checkpoints/last_model.pt
Epoch: 8, Val Loss: 2.2089
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found existing model in Google Drive. Loading...
Resumed from epoch 8, best val_loss: 2.2089


Validation: 100%|██████████| 54/54 [00:04<00:00, 11.21it/s]


New best model saved to Google Drive with val_loss: 2.0831
Epoch 9: Train Loss 2.0575, Val Loss 2.0831


Validation: 100%|██████████| 54/54 [00:04<00:00, 11.64it/s]


New best model saved to Google Drive with val_loss: 1.9787
Epoch 10: Train Loss 1.9172, Val Loss 1.9787


Validation: 100%|██████████| 54/54 [00:04<00:00, 11.71it/s]


New best model saved to Google Drive with val_loss: 1.8821
Epoch 11: Train Loss 1.7974, Val Loss 1.8821
Training completed! Best model saved in: /content/drive/MyDrive/colab_notebooks/model_checkpoints/best_model.pt


##### Check on single batch

In [75]:
# def test_single_batch(model, dataloader, device):
#   model.eval()
#   with torch.no_grad():
#     for batch_idx, (src, tgt) in enumerate(dataloader):
#       if batch_idx >= 1:
#         break

#       src, tgt = src.to(device), tgt.to(device)

#       print(f"Source shape: {src.shape}")
#       print(f"Target shape: {tgt.shape}")

#       src_mask = create_src_mask(src, pad_idx)
#       tgt_mask = create_tgt_mask(tgt[:, :-1], pad_idx)

#       print(f"Source mask shape: {src_mask.shape}")
#       print(f"Target mask shape: {tgt_mask.shape}")

#       # forward pass
#       output = model(src, tgt[:, :-1])
#       print(f"Output shape: {output.shape}")

#       # calculate loss
#       loss = criterion(
#           output.view(-1, output.size(-1)),
#           tgt[:,1:].contiguous().view(-1)
#       )
#       print(f"Test loss: {loss.item():.4f}")

#       break

In [76]:
# # Check
# print("Testing single batch...")
# test_single_batch(model, train_loader, device)

##### Generating Summary

In [77]:
# Summarizer initialization
summarizer = TextSummarizer(
    model=model,
    word2idx=word_2_idx,
    idx2word=idx_2_word,
    device=device
)

In [78]:
# Testing on a single text
sample_text = df['article'][0]
summary = summarizer.summarize(sample_text, method='beam', beam_size=3)
print("Generated Summary:", summary)

Generated Summary: believes believes believes believes believes believes so so against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against against


In [79]:
# Запустите анализ
analyze_generation_issues(model, val_loader, word_2_idx, idx_2_word, device)

=== ANALYZING GENERATION ISSUES ===

--- Sample 1 ---
Source: teenager ben gibson had just seen his middlesbrough side knocked out in the champions league group s...
Target: teenager ben gibson had just seen his middlesbrough side knocked out in the champions league group stage when the email landed from club chairman uncle steve that he had been fired <UNK> that of course was the virtual world of football manager <UNK> on friday boro defender gibson now 22 can help steer his uncles club back towards the big time for
Step 0:
    0.0013 - 'believes'
    0.0013 - 'juventus'
    0.0012 - 'john'
    0.0012 - 'former'
    0.0010 - 'ian'
Step 1:
    0.0151 - 'believes'
    0.0056 - 'so'
    0.0056 - 'sports'
    0.0056 - 'hailed'
    0.0043 - 'became'
Step 2:
    0.0319 - 'believes'
    0.0170 - 'so'
    0.0132 - 'made'
    0.0119 - 'once'
    0.0115 - 'against'
Step 3:
    0.0373 - 'believes'
    0.0207 - 'made'
    0.0183 - 'so'
    0.0131 - 'once'
    0.0124 - 'told'
Step 4:
    0.0340 - 

In [80]:
# Запустите оценку
references, hypotheses = evaluate_model_performance(
    model, val_loader, word_2_idx, idx_2_word, device
)

Evaluating model performance...

Sample 0:
Source: teenager ben gibson had just seen his middlesbrough side knocked out in the cham...
Reference: teenager ben gibson had just seen his middlesbrough side knocked out in the champions league group stage when the email landed from club chairman uncle steve that he had been fired <UNK> that of course was the virtual world of football manager <UNK> on friday boro defender gibson now 22 can help steer his uncles club back towards the big time for
Generated: believes so against about

Sample 0:
Source: a former bomb disposal expert has won his battle with the bulge after shedding h...
Reference: a former bomb disposal expert has won his battle with the bulge after shedding half his bodyweight after undergoing gastric bypass surgery <UNK> steve <UNK> from bradford west yorkshire lost a whopping 16st after the surgery 14 months ago which caused his stomach to shrink to the size of a tennis ball <UNK> the <UNK> said he decided to go through with


KeyboardInterrupt: 

In [ ]:
# Тестируем на разных текстах
sample_texts = [
    df['article'][0],
    df['article'][10],
    df['article'][20]
]

improved_summarizer = ImprovedTextSummarizer(model, word_2_idx, idx_2_word, device)
test_diversity(improved_summarizer, sample_texts)

In [ ]:
# Evaluate on validation set
# blue_score, refs, cands = evaluate_summarization(
#     model, val_loader, word_2_idx, idx_2_word, device, num_samples=5
# )

In [ ]:
# Batch processing
# texts_to_summarize = df['article'][:5]
# summaries = summarizer.batch_summarize(texts_to_summarize, batch_size=4)